# OCTA + OCT 实验结果可视化

选择 **Python (octa)** 内核并从上到下运行。支持 `fusion`、`oct` 和 `octa` 三种模式以及三种二值标签，自动从 checkpoint 读取模型配置。

图中文字为英文。默认优先加载已完成的融合实验；在配置单元格指定 `RUN_NAME` 可切换实验。此 Notebook 只做推理，不启动训练。

In [ ]:
from pathlib import Path
import sys
import json
import csv
import importlib.util
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, Markdown
%matplotlib inline

PROJECT = next((p / "octa_oct_fusion" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "octa_oct_fusion/data.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("请在项目根目录或 octa_oct_fusion 目录中运行 Notebook")
sys.path.insert(0, str(PROJECT.parent / "octa_baseline"))
spec = importlib.util.spec_from_file_location("fusion_visualization_data", PROJECT / "data.py")
data_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_module)
PairedDataset = data_module.PairedDataset
from octa.model import UNet

completed = sorted(p.parent for p in (PROJECT / "runs").glob("*/test_metrics.json")
                   if (p.parent / "best.pt").is_file())
for path in completed:
    print(path.name)
print("Project:", PROJECT)


## 配置

`RUN_NAME=None` 自动选择已完成的实验。也可填入例如 `fusion_3mm_ILM_OPL_GT_Artery_seed42`、`fusion_3mm_ILM_OPL_GT_Vein_seed42` 或 `oct_3mm_ILM_OPL_GT_Capillary_seed42`。

`CASE_IDS=None` 显示所选集合的前几个病例，指定列表则显示相应病例。阈值沿用训练配置；不要用测试集选择阈值。`SAVE_FIGURES=True` 时图片保存到对应实验的 `visualizations/` 目录。

In [ ]:
RUN_NAME = None
SPLIT = "test"  # "val" 或 "test"
CASE_IDS = None  # 例如 ["10451", "10454"]
NUM_CASES = 4
DATA_ROOT_OVERRIDE = None  # 数据迁移时填写新路径
SAVE_FIGURES = True

if RUN_NAME is None:
    preferred = [p for p in completed if p.name.startswith("fusion_")]
    if not completed:
        raise FileNotFoundError("没有已完成实验，请先训练或手动指定 RUN_NAME")
    RUN_DIR = (preferred or completed)[0]
else:
    RUN_DIR = PROJECT / "runs" / RUN_NAME
checkpoint_path = RUN_DIR / "best.pt"
if not checkpoint_path.is_file():
    raise FileNotFoundError(f"找不到最佳模型：{checkpoint_path}")
if SPLIT not in ("val", "test") or NUM_CASES < 1:
    raise ValueError("SPLIT 必须为 val/test，NUM_CASES 必须为正整数")
# 仅加载本项目可信的本地 checkpoint。
checkpoint = torch.load(checkpoint_path, map_location="cpu")
config = checkpoint["args"]
mode = config["mode"]
threshold = float(config.get("threshold", 0.5))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(in_channels=2 if mode == "fusion" else 1, base=config["base_channels"])
model.load_state_dict(checkpoint["model"])
model.to(device).eval()

root = Path(DATA_ROOT_OVERRIDE or config["data_root"])
if not root.is_absolute():
    root = PROJECT.parent / root
if not root.is_dir():
    raise FileNotFoundError(f"数据目录不存在，请设置 DATA_ROOT_OVERRIDE：{root}")
dataset = PairedDataset(root, SPLIT, mode=mode, target=config["target"],
                        scan_size=config["scan_size"], slab=config["slab"], augment=False)
ids = [str(x) for x in CASE_IDS] if CASE_IDS is not None else dataset.ids[:NUM_CASES]
if not ids:
    raise ValueError("请至少选择一个病例")
missing = set(ids) - set(dataset.ids)
if missing:
    raise ValueError(f"病例不属于 {SPLIT} 集合：{sorted(missing)}")
FIG_DIR = RUN_DIR / "visualizations"
if SAVE_FIGURES:
    FIG_DIR.mkdir(exist_ok=True)
print("Run:", RUN_DIR.name)
print("Input mode:", mode, "| Target:", config["target"], "| Split:", SPLIT)
print("Best epoch:", checkpoint["epoch"], "| Threshold:", threshold, "| Device:", device)


## 已保存的测试集指标

以下是训练结束时保存的全测试集 **micro** 指标，不是下方几个病例指标的平均值。若模型仍在训练、尚未生成测试文件，则这里只显示提示。

In [ ]:
metrics_path = RUN_DIR / "test_metrics.json"
if metrics_path.is_file():
    metrics = json.loads(metrics_path.read_text())
    rows = "\n".join(f"| {key} | {value:.6f} |" if isinstance(value, float)
                     else f"| {key} | {value} |" for key, value in metrics.items())
    display(Markdown("| Metric | Value |\n|---|---:|\n" + rows))
    if metrics.get("best_epoch") != checkpoint["epoch"]:
        print("注意：测试记录与当前 checkpoint 轮次不一致，模型可能仍在更新。")
else:
    print("尚无 test_metrics.json；可以查看当前最佳模型的预测，不能视为最终测试结果。")


## 训练与验证曲线

虚线标记当前最佳 checkpoint 的轮次。Loss 与阈值化后的 Dice 不一定同时改善。

In [ ]:
history_path = RUN_DIR / "history.csv"
if history_path.is_file():
    with history_path.open() as stream:
        history = list(csv.DictReader(stream))
    if history:
        epochs = [int(row["epoch"]) for row in history]
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        for ax, metric in zip(axes[:2], ["loss", "dice"]):
            for split in ("train", "val"):
                ax.plot(epochs, [float(row[f"{split}_{metric}"]) for row in history], label=split)
            ax.axvline(checkpoint["epoch"], color="gray", ls="--", label="Best epoch")
            ax.set(xlabel="Epoch", ylabel=metric.title(), title=f"Training / validation {metric}")
            ax.legend()
            ax.grid(alpha=0.2)
        axes[2].plot(epochs, [float(row["learning_rate"]) for row in history])
        axes[2].set(xlabel="Epoch", ylabel="Learning rate", yscale="log", title="Learning rate")
        fig.tight_layout()
        if SAVE_FIGURES:
            fig.savefig(FIG_DIR / "training_curves.png", dpi=160, bbox_inches="tight")
        plt.show()
else:
    print("未找到 history.csv")


## 病例预测与标签对比

每行是同一病例：OCTA、OCT、标签、概率、二值预测、误差图、轮廓叠加图。

误差图：绿色 TP（正确血管）、红色 FP（误检）、蓝色 FN（漏检）、黑色 TN（正确背景）。轮廓图绿色为标签、红色为预测；红色后绘制，重叠处可能覆盖绿色，应结合误差图判断。单模态实验也展示两个输入供参考，实际送入模型的模态由上方 `Input mode` 决定。

In [ ]:
results = []
with torch.inference_mode():
    for case in ids:
        sample = dataset[dataset.ids.index(case)]
        probability = torch.sigmoid(model(sample["image"].unsqueeze(0).to(device)))[0, 0].cpu().numpy()
        target = sample["mask"][0].numpy() > 0
        prediction = probability >= threshold
        tp = int((prediction & target).sum())
        fp = int((prediction & ~target).sum())
        fn = int((~prediction & target).sum())
        dice = 2 * tp / (2 * tp + fp + fn + 1e-8)
        iou = tp / (tp + fp + fn + 1e-8)
        error = np.stack([prediction & ~target, prediction & target, ~prediction & target], axis=-1).astype(float)
        results.append(dict(case=case, pair=sample["pair"].numpy(), target=target,
                            probability=probability, prediction=prediction, error=error,
                            dice=dice, iou=iou))

fig, axes = plt.subplots(len(results), 7, figsize=(23, 3.5 * len(results)), squeeze=False)
for row, result in enumerate(results):
    panels = [result["pair"][0], result["pair"][1], result["target"],
              result["probability"], result["prediction"], result["error"]]
    titles = ["OCTA input", "OCT input", "Ground truth", "Probability",
              f"Prediction t={threshold:g}", "Error map"]
    for col, (panel, title) in enumerate(zip(panels, titles)):
        axes[row, col].imshow(panel, cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_title(f"Case {result['case']} | {title}", fontsize=9)
    overlay = axes[row, 6]
    overlay.imshow(result["pair"][0], cmap="gray", vmin=0, vmax=1)
    for mask, color in ((result["target"], "lime"), (result["prediction"], "red")):
        if mask.any() and not mask.all():
            overlay.contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=0.7)
    overlay.set_title(f"GT green / Pred red\nDice={result['dice']:.3f}, IoU={result['iou']:.3f}", fontsize=9)
    for ax in axes[row]:
        ax.axis("off")
fig.suptitle(f"{mode.upper()} | {config['target']} | {config['scan_size']} | {SPLIT}", fontsize=15)
fig.legend(handles=[Patch(color=color, label=label) for color, label in
                    [("lime", "TP"), ("red", "FP"), ("blue", "FN"), ("black", "TN")]],
           loc="lower center", ncol=4)
fig.tight_layout(rect=[0, 0.035, 1, 0.97])
if SAVE_FIGURES:
    figure_path = FIG_DIR / f"{SPLIT}_prediction_comparison.png"
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    print("Saved:", figure_path)
plt.show()

display(Markdown("| Case | Dice | IoU |\n|---|---:|---:|\n" +
                 "\n".join(f"| {r['case']} | {r['dice']:.4f} | {r['iou']:.4f} |" for r in results)))


## 已完成实验汇总

自动读取本目录中已保存的测试指标。应在相同标签、数据划分与训练协议内比较不同输入模式；此表不代表所有实验都使用了完全相同的配置。

In [ ]:
rows = []
for path in sorted((PROJECT / "runs").glob("*/test_metrics.json")):
    cfg_path = path.parent / "config.json"
    if not cfg_path.is_file():
        continue
    cfg = json.loads(cfg_path.read_text())
    metrics = json.loads(path.read_text())
    rows.append(f"| {path.parent.name} | {cfg['mode']} | {cfg['target']} | " +
                " | ".join(f"{metrics[key]:.4f}" for key in ("dice", "iou", "precision", "recall", "specificity")) + " |")
display(Markdown("| Run | Mode | Target | Dice | IoU | Precision | Recall | Specificity |\n"
                 "|---|---|---|---:|---:|---:|---:|---:|\n" + "\n".join(rows)))
